# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [ ]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

In [ ]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [ ]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

In [ ]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [ ]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])

In [ ]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

In [ ]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

## 4. Перевірка на власних прикладах

In [ ]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
    ("The meeting was postponed until Friday.", "Friday is the new date for the meeting."),
    ("Ukraine won the match 2–1.", "The match ended with Ukraine losing by one goal."),
    ("Please reset my account password.", "I need help changing my profile password."),
    ("The package arrived this morning.", "The parcel has not been delivered yet."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

## Експеримент: 6 епох
Запускається з нової копії checkpoint, тому не продовжує вже натреновану модель.


In [ ]:
model_6 = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
args_6 = TrainingArguments(output_dir="bert-mrpc-6", num_train_epochs=6, learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=64, eval_strategy="epoch", save_strategy="no", fp16=torch.cuda.is_available(), report_to="none", seed=42)
trainer_6 = Trainer(model=model_6, args=args_6, train_dataset=tokenized_datasets["train"], eval_dataset=tokenized_datasets["validation"], data_collator=data_collator, compute_metrics=compute_metrics)
trainer_6.train()
[x for x in trainer_6.state.log_history if "eval_loss" in x]


## Питання та відповіді до лабораторної роботи

### 1. Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?

**Відповідь.** Фактичний час слід записати після запуску в Colab, бо він залежить від GPU, кешу й мережі. Найдовші етапи — інсталяція, завантаження MRPC/BERT і навчання. Типові проблеми: GPU не увімкнено, повільний Hugging Face Hub або несумісні версії бібліотек.

### 2. Як Ви розумієте, що таке токен [CLS]? Яким чином він виконує свою функцію?

**Відповідь.** `[CLS]` — спеціальний перший токен. Через self-attention його прихований стан агрегує інформацію з усієї пари речень, а класифікаційна голова перетворює цей вектор на логіти класів.

### 3. Яке пряме застосування базової моделі [bert-base-uncased](https://huggingface.co/google-bert/bert-base-uncased)?

**Відповідь.** Пряме застосування base BERT у цьому контексті — masked-language modeling: передбачення замаскованого токена `[MASK]`. У демо це показує `fill-mask`. До заміни голова має 30 522 виходи — за словником BERT.

### 4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?

**Відповідь.** Так: стереотипи варто вимірювати контрфактичними тестами, зменшувати збалансованими або контрфактичними даними та перевіряти аудитом. Одного «виправного» промпту недостатньо.

### 5. Для чого в секції "Токенізація пари речень" виводяться числа 0 і 1?

**Відповідь.** Це `token_type_ids`: 0 позначає перше речення разом із `[CLS]`, 1 — друге. Так BERT відрізняє сегменти пари.

### 6. Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?

**Відповідь.** 768 — розмір прихованого представлення BERT-base (`hidden_size`) і число входів голови; 2 — два класи та, відповідно, два логіти.

### 7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?

**Відповідь.** Добре: великий backbone переносить знання з pretraining, а мала голова вчиться конкретній задачі. У цьому демо донавчаються всі ваги; це дорожче і може перенавчитися, тому застосовано малий learning rate та validation.

### 8. Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?

**Відповідь.** Accuracy — частка правильно класифікованих пар на validation split. F1 — гармонійне середнє precision і recall позитивного класу «парафраз».

### 9. Збільшіть кількість епох тренування в 01_bert_head, покажіть (скопіювавши результати) що відбувається і чому.

**Відповідь.** Додано окремий запуск на 6 епох. Він починає з нового checkpoint. Після запуску треба залишити таблицю з `eval_loss`, `eval_accuracy`, `eval_f1`: training loss зазвичай спадає, а validation метрики можуть вийти на плато або погіршитися — це перенавчання.

### 10. Додайте ще 4 перевірки на власних даних в пункті 4 в 01_bert_head.

**Відповідь.** Додано чотири пари: відкладена зустріч, суперечливий результат матчу, reset password та протилежні твердження про доставку.
